# Pokémon Battle Prediction 2026 — Competitive XGBoost Pipeline
**Sapienza University of Rome — Foundations of Data Science Competition**

Questo notebook implementa la pipeline completa, riproducibile e conforme alle regole della competizione basata su **XGBoost** per predire il vincitore finale (`player_won` $\in \{0, 1\}$) delle battaglie Pokémon Generation 2 OU a partire dai primi 30 turni osservati (`battle_timeline`).

### Punti Chiave della Metodologia
1. **Conformità Rigorosa alle Regole**: Utilizza esclusivamente alberi di decisione con gradient boosting (`XGBClassifier`), i file JSONL ufficiali della competizione e la tabella dei tipi di Generazione 2 (`type_chart_gen2.csv`). Nessuna rete neurale, modello pre-addestrato o database esterno.
2. **Ricostruzione Fisica dello Stato Battaglia Gen 2 (`src/battle_features.py`)**:
   - **De-duplicazione della Cache di `move_details`**: Filtra i turni di switch-in volontario in cui il parser del dataset ripete l'ultima mossa memorizzata in cache dal Pokémon, evitando il doppio conteggio delle mosse offensive e di stato.
   - **Ricostruzione Fisica dei Single-KO del Player 2**: Poiché `p2_pokemon_state.status == "fnt"` compare solo nei doppi KO simultanei e al Turno 30 (quando un Pokémon di P2 va KO da solo nei turni 1–29, il sostituto compare immediatamente nello snapshot di fine turno), ricostruiamo i KO singoli di P2 tramite la fisica di Gen 2 (verifica del sostituto intatto, attacco eseguito da P1, efficacia di tipo, danno residuo da status ed *Explosion*).
   - **Catalogo Specie Interno Within-Fold**: Costruito esclusivamente dai dati di training (`p1_team_details` e `p2_lead_details`, 247 specie con 0 incongruenze) per inferire statistiche base e tipi dei Pokémon rivelati da P2.
3. **Sei Famiglie Modulari di Feature + Target Encoding Leak-Free Within-Fold**:
   - **Family A (`team`)**: Statistiche base del team completo di P1, bulk fisico/speciale, speed tier, diversità di tipi e staple del metagame Gen 2 OU.
   - **Family B (`p2_observed`)**: Statistiche del lead e dei Pokémon rivelati da P2, incertezza sulle riserve non rivelate e categorie di mosse osservate.
   - **Family C (`state`)**: Bilancio materiale al Turno 30, differenziale vivi/fainted, HP pesati per recency, gravità degli status (distinguendo *Rest* dal sonno inflitto dal nemico), boost attivi e phazed-absence score.
   - **Family D (`temporal`)**: Traiettorie multi-finestra (`Turns 1–10`, `11–20`, `21–30`, `Last 5`, `Last 10`), pendenza lineare (momentum), cambi di vantaggio e controllo del ritmo.
   - **Family E (`matchup`)**: Matrice di efficacia dei tipi Gen 2 (`type_chart_gen2.csv`), moltiplicatori STAB attivi, miglior counter disponibile nel roster e Speed/KO threat index.
   - **Family F (`interactions`)**: Rapporti non lineari, vantaggio di sopravvivenza aggiustato per HP effettivi e indice di minaccia *Endgame Sweeper* al Turno 30.
   - **`WithinFoldTargetEncoder`**: Calcola i win-rate empirici smoothed (specie, lead matchup e sopravvissuti pesati per HP al Turno 30) rigorosamente all'interno di ciascun fold tramite 5-fold OOF annidati.
4. **Generazione Automatica dell'Unico File `submission.csv`**: Alla fine del notebook viene generato e validato l'unico file **`submission.csv`** pronto per la submission automatica alla challenge su Kaggle.

## Step 1 & 2: Setup Ambiente e Clonazione Repository GitHub
Su Kaggle il notebook clona automaticamente il repository GitHub e lo aggiunge a `sys.path`. In esecuzione locale rileva direttamente la directory del progetto.

In [ ]:
import os
import subprocess
import sys
import time
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")

REPO_URL = os.environ.get("REPO_URL", "https://github.com/danielegabriele17-rgb/predictive-spikes.git")
GIT_REF = os.environ.get("GIT_REF", "main")
CLONE_DIR = Path("/tmp/predictive-spikes") if Path("/kaggle/working").exists() else Path.cwd()

def setup_repository() -> Path:
    for candidate in [Path.cwd(), Path.cwd().parent, CLONE_DIR]:
        if (candidate / "src" / "battle_features.py").exists() and (candidate / "configs" / "experiments.yaml").exists():
            print(f"Repository rilevato in: {candidate.resolve()}")
            return candidate.resolve()

    print(f"Clonazione di {REPO_URL} (ref={GIT_REF}) in {CLONE_DIR}...")
    subprocess.run(["git", "clone", "--depth", "1", "-b", GIT_REF, REPO_URL, str(CLONE_DIR)], check=True)
    return CLONE_DIR.resolve()

REPO_ROOT = setup_repository()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print(f"Repository Root : {REPO_ROOT}")
print(f"Working Dir     : {Path.cwd().resolve()}")

: 

## Step 3: Import dei Moduli della Pipeline (`src/`)

In [ ]:
import numpy as np
import pandas as pd
import xgboost as xgb
import yaml
from sklearn.metrics import accuracy_score, log_loss, roc_auc_score

try:
    from IPython.display import display
except ImportError:
    display = print

from src.battle_features import build_species_catalogue
from src.data_loader import discover_data_dir, load_competition_data
from src.feature_engineering import (
    ALL_FEATURE_FAMILIES,
    WithinFoldTargetEncoder,
    build_feature_dataframe,
)
from src.model import get_feature_importance_df, run_xgb_cv
from src.validation import get_cv_splits

print(f"XGBoost version: {xgb.__version__}")
print(f"NumPy version  : {np.__version__}")
print(f"Pandas version : {pd.__version__}")

## Step 4 & 5: Rilevamento Automatico e Caricamento Dataset
Il data loader (`discover_data_dir` e `load_competition_data`) individua automaticamente `train.jsonl`, `test.jsonl`, `sample_submission.csv` e `type_chart_gen2.csv` sia su Kaggle (`/kaggle/input/...`) sia in locale (`fds-pokemon-battles-prediction-2026/`).

In [ ]:
t_start = time.perf_counter()
DATA_DIR = discover_data_dir()
sample_submission_path = DATA_DIR / "sample_submission.csv"

train_battles, test_battles, sample_sub, type_chart = load_competition_data(custom_dir=DATA_DIR)
y = np.array([int(b["player_won"]) for b in train_battles], dtype=int)

print(f"Directory dati attiva : {DATA_DIR}")
print(f"Battaglie di Training : {len(train_battles):,} | Target positive rate: {y.mean():.4f}")
print(f"Battaglie di Test     : {len(test_battles):,} | Sample submission shape: {sample_sub.shape}")
print(f"Gen 2 Type Chart      : {len(type_chart)} coppie attacco-difesa caricate")

## Step 6: Estrazione delle 6 Famiglie di Feature e Pokédex Interno
Costruiamo il catalogo interno delle specie esclusivamente da `train.jsonl` ed estraiamo tutte le 6 famiglie modulari di feature (`team`, `p2_observed`, `state`, `temporal`, `matchup`, `interactions`) per le 10,000 battaglie di training e le 5,000 battaglie di test.

In [ ]:
t_fe = time.perf_counter()
catalogue = build_species_catalogue(train_battles)
print(f"Catalogo interno delle specie costruito dal training set: {len(catalogue)} specie Pokémon registrate")

X_train_full, train_ids, y_extracted = build_feature_dataframe(
    train_battles,
    families=ALL_FEATURE_FAMILIES,
    catalogue=catalogue,
    type_chart=type_chart,
)
X_test_full, test_ids, _ = build_feature_dataframe(
    test_battles,
    families=ALL_FEATURE_FAMILIES,
    catalogue=catalogue,
    type_chart=type_chart,
)
print(f"Matrici di feature estratte in {time.perf_counter() - t_fe:.2f}s:")
print(f"  X_train_full: {X_train_full.shape}")
print(f"  X_test_full : {X_test_full.shape}")

## Step 7: Caricamento Configurazioni Ottimizzate (`configs/experiments.yaml`)
Carichiamo i sottoinsiemi di feature selezionati (`top140_features`, `top200_features`, `top220_features`, `pruned_features`), gli iperparametri XGBoost ottimizzati con Optuna e i pesi di blending dal file `configs/experiments.yaml`.

In [ ]:
config_path = REPO_ROOT / "configs" / "experiments.yaml"
with open(config_path, "r", encoding="utf-8") as f:
    exp_config = yaml.safe_load(f)

RANDOM_SEED = int(exp_config.get("random_seed", 42))
FINAL_FOLDS = int(exp_config.get("n_splits", 5))
DECISION_THRESHOLD = float(exp_config.get("default_threshold", 0.5))
feature_subsets = exp_config["feature_subsets"]
models_cfg = exp_config["models"]
ens_cfg = exp_config["ensemble"]

print(f"Configurazione caricata da: {config_path}")
for subset_name, cols in feature_subsets.items():
    print(f"  Feature subset [{subset_name:18s}]: {len(cols)} feature")
print("Modelli configurati:", list(models_cfg.keys()))

## Step 8 & 9: Addestramento in 5-Fold Stratified CV e Multi-Model Ensemble Blending
Addestriamo i modelli XGBoost sui 5 fold stratificati (`StratifiedKFold(n_splits=5, shuffle=True, random_state=42)`) calcolando il `WithinFoldTargetEncoder` rigorosamente dentro ciascun fold (zero data leakage) e combiniamo le probabilità OOF e di Test nell'Ensemble finale:
- **`model1_optuna_top200_te`** (`weight = 0.15`): Modello Optuna TPE (`max_depth=4`, `min_child_weight=20`) sulle Top-200 feature + Within-Fold TE.
- **`model2_conservative_depth4_pruned_te`** (`weight = 0.35`): Modello shallow fortemente regolarizzato (`max_depth=4`, `colsample_bytree=0.45`, `reg_lambda=5.0`) su tutte le 512 feature pruned + Within-Fold TE.
- **`model3_obs_only_top220_no_te`** (`weight = 0.10`): Modello *Pure Observation-Only* (Zero Target Encoding, `max_depth=5`) sulle Top-220 feature.
- **`model4_compact_top140_te`** (`weight = 0.40`): Modello compatto ad alto rapporto segnale/rumore (`max_depth=5`, `colsample_bytree=0.60`) sulle Top-140 feature + Within-Fold TE.

In [ ]:
cv_splits = get_cv_splits(train_battles, y, n_splits=FINAL_FOLDS, seed=RANDOM_SEED, group_duplicates=False)

te_folds_train = []
te_folds_val = []
te_folds_test = []
for tr_idx, va_idx in cv_splits:
    enc = WithinFoldTargetEncoder(smoothing=30.0, team_smoothing=20.0)
    tr_b = [train_battles[i] for i in tr_idx]
    va_b = [train_battles[i] for i in va_idx]
    te_folds_train.append(enc.fit_transform_inner_oof(tr_b, y[tr_idx], n_inner_splits=5, seed=RANDOM_SEED))
    te_folds_val.append(enc.transform(va_b))
    te_folds_test.append(enc.transform(test_battles))

fold_map = {(int(tr[0]), int(va[0])): idx for idx, (tr, va) in enumerate(cv_splits)}

def fold_te_fn(tr_idx: np.ndarray, va_idx: np.ndarray):
    idx = fold_map[(int(tr_idx[0]), int(va_idx[0]))]
    return te_folds_train[idx], te_folds_val[idx]

def test_te_fn(tr_idx: np.ndarray, va_idx: np.ndarray):
    idx = fold_map[(int(tr_idx[0]), int(va_idx[0]))]
    return te_folds_test[idx]

oof_probs_map = {}
test_probs_map = {}
cv_results_map = {}

for m_name, m_spec in models_cfg.items():
    cols = feature_subsets[m_spec["feature_subset"]]
    use_te = bool(m_spec.get("use_within_fold_te", True))
    es_rounds = m_spec.get("early_stopping_rounds", 50)
    params = m_spec["xgb_params"]

    cv_res, test_probs = run_xgb_cv(
        X_df=X_train_full[cols],
        y=y,
        splits=cv_splits,
        xgb_params=params,
        experiment_id=m_name,
        early_stopping_rounds=es_rounds,
        fold_feature_fn=fold_te_fn if use_te else None,
        X_test_df=X_test_full[cols],
        test_fold_feature_fn=test_te_fn if use_te else None,
        threshold=DECISION_THRESHOLD,
        seed=RANDOM_SEED,
    )
    cv_results_map[m_name] = cv_res
    oof_probs_map[m_name] = cv_res.oof_probs
    test_probs_map[m_name] = test_probs
    print(
        f"[{m_name:36s}] 5-Fold CV Acc: {cv_res.cv_mean_acc:.4f} +/- {cv_res.cv_std_acc:.4f} "
        f"| ROC-AUC: {cv_res.oof_roc_auc:.4f} | LogLoss: {cv_res.oof_log_loss:.4f} "
        f"| Train-Val Gap: {np.mean(cv_res.fold_train_accs) - cv_res.cv_mean_acc:.4f}"
    )

members = ens_cfg["members"]
weights = np.asarray(ens_cfg["weights"], dtype=float)
weights = weights / np.sum(weights)

blended_oof_probs = sum(w * oof_probs_map[m] for w, m in zip(weights, members))
blended_test_probs = sum(w * test_probs_map[m] for w, m in zip(weights, members))

ens_fold_accs = [
    float(accuracy_score(y[va_idx], (blended_oof_probs[va_idx] >= DECISION_THRESHOLD).astype(int)))
    for _, va_idx in cv_splits
]
final_oof_acc = float(np.mean(ens_fold_accs))
final_oof_std = float(np.std(ens_fold_accs))
final_oof_auc = float(roc_auc_score(y, blended_oof_probs))
final_oof_loss = float(log_loss(y, blended_oof_probs))

print("=" * 80)
print(f"RISULTATO FINALE ENSEMBLE ({FINAL_FOLDS}-Fold x {len(members)} Configurazioni = {FINAL_FOLDS * len(members)} modelli XGBoost)")
print(f"OOF Validation Accuracy : {final_oof_acc:.4f} +/- {final_oof_std:.4f}")
print(f"OOF ROC-AUC             : {final_oof_auc:.4f}")
print(f"OOF Log-Loss            : {final_oof_loss:.4f}")
print("=" * 80)

## Step 10: Diagnostica Feature Importance e Generazione dell'Unico File `submission.csv`

Ispezioniamo le **25 Feature più importanti** del modello principale e generiamo l'**unico file `submission.csv`** nella working directory, verificandone rigorosamente l'allineamento con `sample_submission.csv` per la submission automatica alla challenge.

In [ ]:
# ==============================================================================
# 1. TOP 25 FEATURE PER IMPORTANCE & GRAFICI DIAGNOSTICI (se matplotlib è presente)
# ==============================================================================
cv_m4 = cv_results_map["model4_compact_top140_te"]
fi_df = get_feature_importance_df(cv_m4.models, cv_m4.feature_names)
print("Top 25 Feature per Importance (Modello Principale):")
print(fi_df.head(25).to_string(index=False))

try:
    %pip install matplotlib

    fig, axes = plt.subplots(1, 2, figsize=(15, 6))
    top_n = min(25, len(fi_df))
    top_fi = fi_df.iloc[:top_n][::-1]
    axes[0].barh(top_fi["feature"], top_fi["importance"], color="#2b7bba")
    axes[0].set_title(f"Top {top_n} Feature per Importance (Modello Ottimizzato)")
    axes[0].set_xlabel("Importance Media")

    fold_ids = [f"Fold {i}" for i in range(1, FINAL_FOLDS + 1)]
    axes[1].bar(fold_ids, ens_fold_accs, color="#2e8b57", alpha=0.85, label="Fold Val Acc")
    axes[1].axhline(final_oof_acc, color="#d9534f", linestyle="--", linewidth=2, label=f"Blended OOF Acc ({final_oof_acc:.4f})")
    axes[1].set_ylim(min(ens_fold_accs) - 0.03, max(ens_fold_accs) + 0.03)
    axes[1].set_title(f"Accuratezza di Validazione sui {FINAL_FOLDS} Fold")
    axes[1].set_ylabel("Accuracy")
    axes[1].tick_params(axis="x", rotation=30)
    axes[1].legend(loc="best")

    plt.tight_layout()
    plt.show()
except ImportError:
    pass

# ==============================================================================
# 2. GENERAZIONE DELL'UNICO FILE DI SUBMISSION PER KAGGLE (submission.csv)
# ==============================================================================
def create_single_kaggle_submission(test_Ids, test_probabilities, threshold=0.50, output_filename="submission.csv"):
    """
    Genera l'unico file di submission rispettando rigorosamente il formato di sample_submission.csv
    (1 -> Player 1 wins, 0 -> Player 1 loses; mantiene ogni battle_id accoppiato alla sua predizione).
    """
    binary_preds = (test_probabilities >= threshold).astype(int)

    if sample_submission_path.exists():
        sub = pd.read_csv(sample_submission_path)
    else:
        sub = pd.DataFrame({"battle_id": test_Ids, "player_won": 0})

    pred_map = pd.Series(binary_preds, index=test_Ids)
    sub["player_won"] = sub["battle_id"].map(pred_map)

    assert list(sub.columns) == ["battle_id", "player_won"], f"Colonne errate: {list(sub.columns)}"
    assert len(sub) == len(test_Ids), f"Numero di righe errato: {len(sub)} vs {len(test_Ids)}"
    assert not sub["player_won"].isna().any(), "Errore: presenti valori NaN nelle predizioni!"
    sub["player_won"] = sub["player_won"].astype(int)
    assert set(sub["player_won"].unique()).issubset({0, 1}), "Errore: le predizioni devono essere binarie (0 o 1)!"

    sub.to_csv(output_filename, index=False)
    print("=" * 72)
    print(f"File '{output_filename}' generato con successo!")
    print(f"Totale battaglie predette : {len(sub)}")
    print(f"Vittorie Player 1 (1)     : {(sub['player_won'] == 1).sum()} ({sub['player_won'].mean()*100:.2f}%)")
    print(f"Vittorie Player 2 (0)     : {(sub['player_won'] == 0).sum()} ({(1 - sub['player_won'].mean())*100:.2f}%)")
    print("=" * 72)
    return sub

submission_df = create_single_kaggle_submission(
    test_Ids=test_ids,
    test_probabilities=blended_test_probs,
    threshold=DECISION_THRESHOLD,
    output_filename="submission.csv",
)

display(submission_df.head(10))